# 🔮 LSTM Spending Predictor — Training Notebook
### Intelligent Expense Tracker — Final Year Project

This notebook:
1. Generates 365 days of realistic synthetic spending data
2. Trains an LSTM model to predict future daily spending
3. Trains an ARIMA model for comparison
4. Evaluates both using RMSE and MAPE
5. Saves the trained LSTM model as `.h5` for Flask backend

**Enable GPU: Runtime → Change runtime type → T4 GPU**

**Run each cell top to bottom. Takes ~3 minutes total.**

## Cell 1 — Install & Import Libraries

In [ ]:
!pip install tensorflow statsmodels pandas numpy matplotlib scikit-learn -q

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import warnings
warnings.filterwarnings('ignore')

from datetime import datetime, timedelta
import random

# TensorFlow / Keras
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Input
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam

# Stats
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.stattools import adfuller

random.seed(42)
np.random.seed(42)
tf.random.set_seed(42)

print('✅ All libraries imported!')
print(f'   TensorFlow version: {tf.__version__}')
print(f'   GPU available: {len(tf.config.list_physical_devices("GPU")) > 0}')

## Cell 2 — Generate Synthetic Spending Time Series

In [ ]:
# ─────────────────────────────────────────────────────────────
# Generate 365 days of realistic Indian daily spending data
# Includes: weekly patterns, monthly salary cycles, seasonal spikes
# ─────────────────────────────────────────────────────────────

def generate_spending_timeseries(days=365, base_daily=500):
    """
    Simulates realistic Indian personal spending patterns.
    base_daily: average daily spend in INR (~₹500)
    """
    dates = [datetime(2024, 1, 1) + timedelta(days=i) for i in range(days)]
    amounts = []

    for i, date in enumerate(dates):
        amount = base_daily

        # 1. Weekly pattern — weekends have 60% higher spending
        if date.weekday() >= 5:  # Saturday=5, Sunday=6
            amount *= random.uniform(1.4, 1.8)
        else:
            amount *= random.uniform(0.7, 1.1)

        # 2. Monthly salary effect — high spending at start and end of month
        day_of_month = date.day
        if 1 <= day_of_month <= 5:    # just received salary
            amount *= random.uniform(1.3, 1.7)
        elif 25 <= day_of_month <= 31:  # month end, buying essentials
            amount *= random.uniform(1.1, 1.4)
        else:
            amount *= random.uniform(0.8, 1.1)

        # 3. Festival seasons — Diwali (Oct-Nov), Christmas (Dec), Holi (Mar)
        month, day = date.month, date.day
        if (month == 10 and day >= 20) or (month == 11 and day <= 10):  # Diwali
            amount *= random.uniform(1.5, 2.5)
        elif month == 12 and day >= 20:  # Christmas/New Year
            amount *= random.uniform(1.3, 2.0)
        elif month == 3 and 20 <= day <= 28:  # Holi
            amount *= random.uniform(1.2, 1.6)
        elif month == 8 and 14 <= day <= 16:  # Independence Day
            amount *= random.uniform(1.1, 1.4)

        # 4. Occasional zero days (stayed home, no spending)
        if random.random() < 0.05:  # 5% chance of zero spend day
            amount = 0

        # 5. Random noise
        amount += random.gauss(0, 50)
        amount = max(0, round(amount, 2))  # no negative spending

        amounts.append(amount)

    df = pd.DataFrame({'date': dates, 'amount': amounts})
    df.set_index('date', inplace=True)
    return df


df = generate_spending_timeseries(days=365, base_daily=500)

print('✅ Time series generated!')
print(f'   Date range  : {df.index[0].date()} → {df.index[-1].date()}')
print(f'   Total days  : {len(df)}')
print(f'   Total spent : ₹{df["amount"].sum():,.2f}')
print(f'   Daily avg   : ₹{df["amount"].mean():.2f}')
print(f'   Daily min   : ₹{df["amount"].min():.2f}')
print(f'   Daily max   : ₹{df["amount"].max():.2f}')
df.head(10)

## Cell 3 — Visualize the Time Series

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 12))
fig.suptitle('Synthetic Spending Data Analysis', fontsize=16, fontweight='bold')

# Plot 1: Full year daily spending
axes[0].plot(df.index, df['amount'], color='#3b82f6', linewidth=0.8, alpha=0.8)
axes[0].fill_between(df.index, df['amount'], alpha=0.2, color='#3b82f6')
axes[0].set_title('Daily Spending — Full Year 2024')
axes[0].set_ylabel('Amount (₹)')
axes[0].xaxis.set_major_formatter(mdates.DateFormatter('%b'))
axes[0].grid(True, alpha=0.3)

# Plot 2: 30-day rolling average
rolling = df['amount'].rolling(window=30).mean()
axes[1].plot(df.index, df['amount'], color='#94a3b8', linewidth=0.6, alpha=0.5, label='Daily')
axes[1].plot(df.index, rolling, color='#f97316', linewidth=2, label='30-Day Rolling Avg')
axes[1].set_title('Daily vs 30-Day Rolling Average')
axes[1].set_ylabel('Amount (₹)')
axes[1].legend()
axes[1].xaxis.set_major_formatter(mdates.DateFormatter('%b'))
axes[1].grid(True, alpha=0.3)

# Plot 3: Monthly totals
monthly = df.resample('ME')['amount'].sum()
axes[2].bar(monthly.index, monthly.values, color='#22c55e', width=20, alpha=0.8)
axes[2].set_title('Monthly Total Spending')
axes[2].set_ylabel('Total Amount (₹)')
axes[2].xaxis.set_major_formatter(mdates.DateFormatter('%b'))
for i, (date, val) in enumerate(monthly.items()):
    axes[2].text(date, val + 100, f'₹{val/1000:.1f}k', ha='center', fontsize=8)
axes[2].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('spending_timeseries.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Time series plots saved!')

## Cell 4 — Prepare Data for LSTM

In [ ]:
# ─────────────────────────────────────────────────────────────
# LSTM requires data in sliding window sequences
# We use past 30 days to predict the next day
# ─────────────────────────────────────────────────────────────

WINDOW_SIZE = 30  # use last 30 days as input
FORECAST_DAYS = 30  # predict next 30 days

# Normalize data to [0, 1] — LSTM works best with normalized data
scaler = MinMaxScaler(feature_range=(0, 1))
scaled_data = scaler.fit_transform(df[['amount']])

print(f'Original data range : ₹{df["amount"].min():.2f} – ₹{df["amount"].max():.2f}')
print(f'Scaled data range   : {scaled_data.min():.4f} – {scaled_data.max():.4f}')

def create_sequences(data, window_size):
    """Convert flat array into (X, y) sliding window sequences."""
    X, y = [], []
    for i in range(len(data) - window_size):
        X.append(data[i : i + window_size, 0])
        y.append(data[i + window_size, 0])
    return np.array(X), np.array(y)

X, y = create_sequences(scaled_data, WINDOW_SIZE)

# Reshape X for LSTM: (samples, timesteps, features)
X = X.reshape(X.shape[0], X.shape[1], 1)

# 80/20 train-test split (preserve time order — no shuffling!)
split = int(len(X) * 0.8)
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

print(f'\n✅ Sequences created:')
print(f'   Window size  : {WINDOW_SIZE} days')
print(f'   Total seqs   : {len(X)}')
print(f'   Training     : {len(X_train)} sequences')
print(f'   Testing      : {len(X_test)} sequences')
print(f'   X shape      : {X_train.shape}  (samples, timesteps, features)')

## Cell 5 — Build & Train the LSTM Model

In [ ]:
# ─────────────────────────────────────────────────────────────
# LSTM Architecture (matches research paper description)
# ─────────────────────────────────────────────────────────────

def build_lstm_model(window_size):
    model = Sequential([
        Input(shape=(window_size, 1)),
        LSTM(64, return_sequences=True),
        Dropout(0.2),
        LSTM(50, return_sequences=False),
        Dropout(0.2),
        Dense(25, activation='relu'),
        Dense(1)  # single output: next day's spending
    ])
    model.compile(
        optimizer=Adam(learning_rate=0.001),
        loss='mean_squared_error',
        metrics=['mae']
    )
    return model

lstm_model = build_lstm_model(WINDOW_SIZE)
lstm_model.summary()

In [ ]:
# ─────────────────────────────────────────────────────────────
# TRAIN THE LSTM
# EarlyStopping prevents overfitting
# ReduceLROnPlateau adjusts learning rate when stuck
# ─────────────────────────────────────────────────────────────

callbacks = [
    EarlyStopping(
        monitor='val_loss',
        patience=15,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=7,
        min_lr=1e-6,
        verbose=1
    )
]

print('🚀 Training LSTM model...')
history = lstm_model.fit(
    X_train, y_train,
    epochs=100,
    batch_size=16,
    validation_split=0.15,
    callbacks=callbacks,
    verbose=1
)

print(f'\n✅ Training complete!')
print(f'   Epochs trained : {len(history.history["loss"])}')
print(f'   Final train loss : {history.history["loss"][-1]:.6f}')
print(f'   Final val loss   : {history.history["val_loss"][-1]:.6f}')

## Cell 6 — Plot Training History

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('LSTM Training History', fontsize=14, fontweight='bold')

epochs_range = range(1, len(history.history['loss']) + 1)

# Loss curves
axes[0].plot(epochs_range, history.history['loss'], label='Train Loss', color='#3b82f6')
axes[0].plot(epochs_range, history.history['val_loss'], label='Val Loss', color='#f97316')
axes[0].set_title('Loss (MSE)')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# MAE curves
axes[1].plot(epochs_range, history.history['mae'], label='Train MAE', color='#22c55e')
axes[1].plot(epochs_range, history.history['val_mae'], label='Val MAE', color='#ec4899')
axes[1].set_title('Mean Absolute Error')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('MAE')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('lstm_training_history.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Training history saved!')

## Cell 7 — Evaluate LSTM on Test Data

In [ ]:
# ─────────────────────────────────────────────────────────────
# EVALUATE: predict on test set, inverse transform, compute metrics
# ─────────────────────────────────────────────────────────────

# Predict
lstm_preds_scaled = lstm_model.predict(X_test)

# Inverse transform back to ₹ values
lstm_preds = scaler.inverse_transform(lstm_preds_scaled)
y_test_actual = scaler.inverse_transform(y_test.reshape(-1, 1))

# Metrics
lstm_rmse = np.sqrt(mean_squared_error(y_test_actual, lstm_preds))
lstm_mape = mean_absolute_percentage_error(y_test_actual, lstm_preds) * 100

print('📊 LSTM Evaluation on Test Set:')
print(f'   RMSE : ₹{lstm_rmse:.2f}')
print(f'   MAPE : {lstm_mape:.2f}%')

# Plot predictions vs actual
test_dates = df.index[split + WINDOW_SIZE:]

plt.figure(figsize=(14, 6))
plt.plot(test_dates, y_test_actual, color='#3b82f6', linewidth=1.5, label='Actual Spending')
plt.plot(test_dates, lstm_preds, color='#f97316', linewidth=1.5, linestyle='--', label='LSTM Prediction')
plt.title('LSTM: Actual vs Predicted Daily Spending (Test Set)', fontsize=13)
plt.xlabel('Date')
plt.ylabel('Amount (₹)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%d %b'))
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('lstm_predictions.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Prediction plot saved!')

## Cell 8 — Train ARIMA & Compare with LSTM

In [ ]:
# ─────────────────────────────────────────────────────────────
# ARIMA model — traditional baseline for comparison
# ─────────────────────────────────────────────────────────────

# Check stationarity with ADF test
adf_result = adfuller(df['amount'].dropna())
print(f'ADF Test p-value: {adf_result[1]:.4f}')
print(f'Series is {"stationary" if adf_result[1] < 0.05 else "non-stationary (will difference)"}')

# Use same train/test split as LSTM
total_split = split + WINDOW_SIZE
train_series = df['amount'][:total_split]
test_series  = df['amount'][total_split:]

print(f'\n🔵 Training ARIMA model...')
print(f'   Train size: {len(train_series)} days')
print(f'   Test size : {len(test_series)} days')

# ARIMA(5,1,2) — p=5 past obs, d=1 difference, q=2 MA terms
arima_model = ARIMA(train_series, order=(5, 1, 2))
arima_fit = arima_model.fit()

# Forecast for test period
arima_preds = arima_fit.forecast(steps=len(test_series))
arima_preds = np.maximum(arima_preds, 0)  # no negative spending

arima_rmse = np.sqrt(mean_squared_error(test_series, arima_preds))
arima_mape = mean_absolute_percentage_error(test_series, arima_preds) * 100

print(f'\n📊 ARIMA Evaluation:')
print(f'   RMSE : ₹{arima_rmse:.2f}')
print(f'   MAPE : {arima_mape:.2f}%')

# ── Comparison Summary ────────────────────────────────────
print('\n' + '='*45)
print(f'{"Model":<15} {"RMSE (₹)":>12} {"MAPE (%)":>12}')
print('='*45)
print(f'{"LSTM":<15} {lstm_rmse:>12.2f} {lstm_mape:>12.2f}')
print(f'{"ARIMA":<15} {arima_rmse:>12.2f} {arima_mape:>12.2f}')
print('='*45)
winner = 'LSTM' if lstm_rmse < arima_rmse else 'ARIMA'
print(f'🏆 Better model: {winner}')

In [ ]:
# Side-by-side comparison plot
plt.figure(figsize=(14, 7))
plt.plot(test_dates, test_series.values, color='#3b82f6', linewidth=2, label='Actual', zorder=3)
plt.plot(test_dates, lstm_preds.flatten(), color='#f97316', linewidth=1.5, linestyle='--', label=f'LSTM (MAPE={lstm_mape:.1f}%)')
plt.plot(test_dates, arima_preds.values, color='#a855f7', linewidth=1.5, linestyle=':', label=f'ARIMA (MAPE={arima_mape:.1f}%)')
plt.title('LSTM vs ARIMA — Spending Forecast Comparison', fontsize=13)
plt.xlabel('Date')
plt.ylabel('Amount (₹)')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%d %b'))
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('lstm_vs_arima.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Comparison plot saved!')

## Cell 9 — Generate 30-Day Future Forecast

In [ ]:
# ─────────────────────────────────────────────────────────────
# Use the last 30 days of data to forecast NEXT 30 days
# This is what Flask /api/predict endpoint will do
# ─────────────────────────────────────────────────────────────

def forecast_next_n_days(model, scaler, last_sequence, n_days=30):
    """
    Iteratively predict n_days into the future.
    Uses each prediction as input for the next step.
    """
    predictions = []
    current_seq = last_sequence.copy()  # shape (30, 1)

    for _ in range(n_days):
        # Reshape for LSTM: (1, window_size, 1)
        x_input = current_seq.reshape(1, WINDOW_SIZE, 1)
        # Predict next day
        next_pred = model.predict(x_input, verbose=0)[0][0]
        predictions.append(next_pred)
        # Slide window: drop oldest, add new prediction
        current_seq = np.append(current_seq[1:], [[next_pred]], axis=0)

    # Inverse transform to ₹
    preds_array = np.array(predictions).reshape(-1, 1)
    return scaler.inverse_transform(preds_array).flatten()


# Get last 30 days of actual data as starting sequence
last_30_scaled = scaled_data[-WINDOW_SIZE:]

# Forecast
future_amounts = forecast_next_n_days(lstm_model, scaler, last_30_scaled, n_days=30)
future_amounts = np.maximum(future_amounts, 0)  # no negatives

# Create future dates
last_date = df.index[-1]
future_dates = [last_date + timedelta(days=i+1) for i in range(30)]

future_df = pd.DataFrame({'date': future_dates, 'predicted_amount': future_amounts})

print('📅 30-Day Forecast:')
print(future_df.to_string(index=False))
print(f'\n💰 Total predicted for next 30 days: ₹{future_amounts.sum():,.2f}')
print(f'📊 Average daily predicted: ₹{future_amounts.mean():.2f}')

In [ ]:
# Forecast visualization
last_60_actual = df['amount'].tail(60)

plt.figure(figsize=(14, 6))

# Actual (past 60 days)
plt.plot(last_60_actual.index, last_60_actual.values,
         color='#3b82f6', linewidth=2, label='Actual (Last 60 Days)')

# Future predictions
plt.plot(future_dates, future_amounts,
         color='#f97316', linewidth=2, linestyle='--', label='LSTM Forecast (Next 30 Days)')

# Prediction uncertainty band
plt.fill_between(future_dates,
                 future_amounts * 0.85,
                 future_amounts * 1.15,
                 alpha=0.15, color='#f97316', label='±15% Confidence Band')

# "Today" marker
plt.axvline(x=last_date, color='red', linestyle=':', linewidth=2, label='Today')
plt.text(last_date, plt.ylim()[1]*0.95, ' Today', color='red', fontsize=10)

plt.title('30-Day Spending Forecast', fontsize=14, fontweight='bold')
plt.xlabel('Date')
plt.ylabel('Amount (₹)')
plt.legend(fontsize=10)
plt.grid(True, alpha=0.3)
plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%d %b'))
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('30day_forecast.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ 30-day forecast plot saved!')

## Cell 10 — Save All Model Files

In [ ]:
import os
import joblib

os.makedirs('models', exist_ok=True)

# Save LSTM model
lstm_model.save('models/lstm_model.h5')
print('✅ Saved: models/lstm_model.h5')

# Save scaler (needed to normalize/denormalize in Flask)
joblib.dump(scaler, 'models/lstm_scaler.pkl')
print('✅ Saved: models/lstm_scaler.pkl')

# Save model config as JSON (window size, etc.)
import json
config = {
    'window_size': WINDOW_SIZE,
    'forecast_days': FORECAST_DAYS,
    'lstm_rmse': float(lstm_rmse),
    'lstm_mape': float(lstm_mape),
    'arima_rmse': float(arima_rmse),
    'arima_mape': float(arima_mape),
    'trained_on': str(datetime.now())
}
with open('models/lstm_config.json', 'w') as f:
    json.dump(config, f, indent=2)
print('✅ Saved: models/lstm_config.json')

# Save sample forecast as CSV (for testing Flask)
future_df.to_csv('models/sample_forecast.csv', index=False)
print('✅ Saved: models/sample_forecast.csv')

print(f'\n📦 All files saved in /models folder')
print(json.dumps(config, indent=2))

## Cell 11 — Download Everything

In [ ]:
import shutil
from google.colab import files

# Zip models folder
shutil.make_archive('expense_lstm_models', 'zip', 'models')
print('✅ Zipped all model files → expense_lstm_models.zip')

# Download
files.download('expense_lstm_models.zip')
files.download('lstm_vs_arima.png')
files.download('30day_forecast.png')
files.download('lstm_training_history.png')

print('\n📥 Downloads started!')
print('\n📌 NEXT STEPS:')
print('   1. Extract expense_lstm_models.zip')
print('   2. Copy lstm_model.h5 and lstm_scaler.pkl')
print('   3. Paste into Flask backend/models/ folder')
print('   4. Your predictor.py will load these automatically')
print('\n📊 Final Model Performance Summary:')
print(f'   LSTM  → RMSE: ₹{lstm_rmse:.2f} | MAPE: {lstm_mape:.2f}%')
print(f'   ARIMA → RMSE: ₹{arima_rmse:.2f} | MAPE: {arima_mape:.2f}%')